# Séance 1
## Chargement des données

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import os

In [2]:
import sys
sys.path.append("../pipelines")
from aq import *

print("Lot :", BATCH, "| date de référence :", DATE_REF)
print(lire("SELECT VERSION() AS version, @@collation_server AS collation"))

Lot : L-2026-10-01 | date de référence : 2026-10-01
                  version          collation
0  11.2.6-MariaDB-ubu2204  utf8mb4_nopad_bin


## US-01

In [3]:
# US-01 - Inventaire des sources cif et crm

# 1.1 Tables, colonnes et clés primaires
cols = lire("""
    SELECT table_schema AS base, table_name AS `table`, COUNT(*) AS colonnes
    FROM information_schema.columns
    WHERE table_schema IN ('cif', 'crm')
    GROUP BY table_schema, table_name
""")

cles = lire("""
    SELECT table_schema AS base, table_name AS `table`,
           GROUP_CONCAT(column_name ORDER BY ordinal_position) AS cle_primaire
    FROM information_schema.key_column_usage
    WHERE table_schema IN ('cif', 'crm') AND constraint_name = 'PRIMARY'
    GROUP BY table_schema, table_name
""")

inventaire = cols.merge(cles, how="left", on=["base", "table"])

# 1.2 Comptage exact des lignes
inventaire["lignes"] = [
    int(lire(f"SELECT COUNT(*) AS n FROM {b}.{t}").n[0])
    for b, t in zip(inventaire.base, inventaire.table)
]

print(inventaire.sort_values(["base", "table"]).to_string(index=False))

# 1.3 Tables sans clé primaire
print(
    "\nTables sans clé primaire :",
    inventaire[inventaire.cle_primaire.isna()][["base", "table"]].values.tolist()
)

base        table  colonnes cle_primaire  lignes
 cif      clients        10      column0   40000
 cif       compte         4      column0   45000
 cif         pays         2    code_pays     250
 crm       client        13    client_id  100000
 crm      contrat         9   contrat_id  100000
 crm     sinistre        10  sinistre_id  100000
 crm type_contrat         3    code_type      12

Tables sans clé primaire : []


## US-02

In [4]:
# US-02
tables_bronze = lire("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
    ORDER BY table_name
""")

print(tables_bronze.to_string(index=False))

      table_name
     cif_clients
      cif_compte
        cif_pays
      crm_client
     crm_contrat
    crm_sinistre
crm_type_contrat


In [5]:
%run ../pipelines/01_ingest_bronze.py

In [6]:
for table in tables_bronze["table_name"]:
    resultat = lire(f"""
        SELECT COUNT(*) AS nb_lignes
        FROM bronze.{table}
        WHERE _batch_id = :batch
    """, batch=BATCH)

    print(f"{table} : {resultat['nb_lignes'].iloc[0]} lignes")

cif_clients : 40000 lignes
cif_compte : 45000 lignes
cif_pays : 250 lignes
crm_client : 100000 lignes
crm_contrat : 100000 lignes
crm_sinistre : 100000 lignes
crm_type_contrat : 12 lignes


In [7]:
lire("""
    SELECT COUNT(*) AS nb_lignes,
           SUM(_ingested_at IS NULL) AS dates_manquantes
    FROM bronze.crm_client
    WHERE _batch_id = :batch
""", batch=BATCH)

,nb_lignes,dates_manquantes
0,100000,0


In [8]:
# US-02 - Vérification d'idempotence : comptage avant relance

avant = lire("""
    SELECT COUNT(*) AS nb
    FROM bronze.crm_client
    WHERE _batch_id = :b
""", b=BATCH)

print("Avant :", avant.iloc[0]["nb"])

Avant : 100000


In [9]:
# US-02 - Vérification d'idempotence : relance du même lot

%run ../pipelines/01_ingest_bronze.py

apres = lire("""
    SELECT COUNT(*) AS nb
    FROM bronze.crm_client
    WHERE _batch_id = :b
""", b=BATCH)

print("Après :", apres.iloc[0]["nb"])
print("Écart :", apres.iloc[0]["nb"] - avant.iloc[0]["nb"])

Après : 100000
Écart : 0


## US-03

In [10]:
# US-03 : Porte B - Réconciliation source -> bronze (RQ-021)

def somme_controle(base, table, ou=""):
    colonnes = lire("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :s
          AND table_name = :t
          AND column_name NOT LIKE '\\_%'
          AND column_name <> 'bronze_row_id'
        ORDER BY ordinal_position
    """, s=base, t=table).column_name.tolist()

    liste = ", ".join(colonnes)

    r = lire(f"""
        SELECT COUNT(*) AS n,
               COALESCE(SUM(CRC32(CONCAT_WS('|', {liste}))), 0) AS crc
        FROM {base}.{table} {ou}
    """)

    return int(r.n[0]), int(r.crc[0])

In [11]:
somme_controle("crm", "client")
somme_controle("bronze", "crm_client", f"WHERE _batch_id = '{BATCH}'")

(100000, 214577921396822)

In [12]:
print("Source :", somme_controle("crm", "client"))
print("Bronze :", somme_controle("bronze", "crm_client", f"WHERE _batch_id = '{BATCH}'"))

Source : (100000, 214577921396822)
Bronze : (100000, 214577921396822)


In [13]:
SOURCES = [("crm", "client", "crm_client"), ("crm", "contrat", "crm_contrat"), ("crm", "type_contrat", "crm_type_contrat"), ("crm", "sinistre", "crm_sinistre"),
           ("cif", "clients", "cif_clients"), ("cif", "compte", "cif_compte"), ("cif", "pays", "cif_pays")]

In [14]:
porte_b = []
for base, table, cible in SOURCES:
    n_src, crc_src = somme_controle(base, table)
    n_brz, crc_brz = somme_controle("bronze", cible, f"WHERE _batch_id = '{BATCH}'")
    ecarts = abs(n_src - n_brz) + int(crc_src != crc_brz)
    porte_b.append(mesurer("BRONZE", "RQ-021", cible, n_src, ecarts))
    # RQ-022 : chaque ligne porte lot, source, horodatage et empreinte
    meta = lire(f"""SELECT COUNT(*) AS n, COALESCE(SUM(_batch_id IS NULL OR _batch_id = '' OR _source_system IS NULL OR _source_system = ''
               OR _ingested_at IS NULL OR _row_hash IS NULL OR _row_hash = ''), 0) AS ko FROM bronze.{cible} WHERE _batch_id = '{BATCH}'""")
    porte_b.append(mesurer("BRONZE", "RQ-022", cible, int(meta.n[0]), int(meta.ko[0])))
print(pd.DataFrame(porte_b).sort_values(["rule_id", "table"]).to_string(index=False))   # aucun écart attendu

rule_id couche            table  testees  echecs  conformite_pct
 RQ-021 BRONZE      cif_clients    40000       0           100.0
 RQ-021 BRONZE       cif_compte    45000       0           100.0
 RQ-021 BRONZE         cif_pays      250       0           100.0
 RQ-021 BRONZE       crm_client   100000       0           100.0
 RQ-021 BRONZE      crm_contrat   100000       0           100.0
 RQ-021 BRONZE     crm_sinistre   100000       0           100.0
 RQ-021 BRONZE crm_type_contrat       12       0           100.0
 RQ-022 BRONZE      cif_clients    40000       0           100.0
 RQ-022 BRONZE       cif_compte    45000       0           100.0
 RQ-022 BRONZE         cif_pays      250       0           100.0
 RQ-022 BRONZE       crm_client   100000       0           100.0
 RQ-022 BRONZE      crm_contrat   100000       0           100.0
 RQ-022 BRONZE     crm_sinistre   100000       0           100.0
 RQ-022 BRONZE crm_type_contrat       12       0           100.0


In [15]:
# Test avec des écarts
executer("DROP TABLE IF EXISTS bronze.crm_client_test")
executer("CREATE TABLE bronze.crm_client_test LIKE bronze.crm_client")
executer("INSERT INTO bronze.crm_client_test SELECT * FROM bronze.crm_client")

executer("DELETE FROM bronze.crm_client_test LIMIT 1")
n_t, crc_t = somme_controle("bronze", "crm_client_test")
n_s, crc_s = somme_controle("crm", "client")
print("copie contre source : écarts =", abs(n_s - n_t) + int(crc_s != crc_t), "(0 tant que la copie n'est pas altérée)")
executer("DROP TABLE bronze.crm_client_test")

copie contre source : écarts = 2 (0 tant que la copie n'est pas altérée)


In [16]:
# US-03 - Vérification de l'historique des mesures RQ-021 et RQ-022

lire("SELECT COUNT(*) AS nb_mesures FROM dq.dq_history")

,nb_mesures
0,14


In [46]:
# US-03 - Preuve de l'enregistrement de RQ-021 dans dq.dq_history

lire("""
    SELECT *
    FROM dq.dq_history
    WHERE rule_id = 'RQ-021'
      AND couche = 'BRONZE'
""")

,id,batch_id,couche,rule_id,table_cible,nb_testees,nb_echecs,taux_conformite,mesure_le
0,43,L-2026-10-01,BRONZE,RQ-021,crm_client,100000,0,100.000,2026-10-28 21:24:14
1,45,L-2026-10-01,BRONZE,RQ-021,crm_contrat,100000,0,100.000,2026-10-28 21:24:15
2,47,L-2026-10-01,BRONZE,RQ-021,crm_type_contrat,12,0,100.000,2026-10-28 21:24:15
3,49,L-2026-10-01,BRONZE,RQ-021,crm_sinistre,100000,0,100.000,2026-10-28 21:24:16
4,51,L-2026-10-01,BRONZE,RQ-021,cif_clients,40000,0,100.000,2026-10-28 21:24:16
5,53,L-2026-10-01,BRONZE,RQ-021,cif_compte,45000,0,100.000,2026-10-28 21:24:17
6,55,L-2026-10-01,BRONZE,RQ-021,cif_pays,250,0,100.000,2026-10-28 21:24:17


## US-06 — Profilage complet de bronze.crm_client

In [17]:
# US-06 - Étape 1 : Charger les clients de Bronze

df = lire("SELECT * FROM bronze.crm_client WHERE _batch_id = :b", b=BATCH)

print(f"[OK] {len(df)} lignes chargées")

[OK] 100000 lignes chargées


In [18]:
# US-06 - Étape 2 : Génération du rapport YData-Profiling

from ydata_profiling import ProfileReport

# Profilage complet de la table crm.client (100 000 lignes)
profile = ProfileReport(
    df,
    title="AssurConnect — Profilage crm.client (100 000 lignes)",
    explorative=True
)
profile.to_file("/home/jovyan/work/reports/ydata_profiling_client.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 18/18 [00:13<00:00,  1.29it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
# US-06 - Étape 4 : Compter les codes postaux français à 4 chiffres (A01)

print(
    "Codes postaux FR à 4 chiffres (A01) :",
    ((df["cp"].str.len() == 4) & (df["pays"] == "FR")).sum()
)

Codes postaux FR à 4 chiffres (A01) : 1456


In [20]:
# US-06 - Étape 5 : Vérification des codes pays (A02)
# Code du professeur, avec la connexion existante lire()

ref_pays = lire("SELECT code_pays FROM silver.ref_pays")

print(
    "Pays hors référentiel (A02, A18) :",
    (~df["pays"].isin(ref_pays["code_pays"])).sum()
)

Pays hors référentiel (A02, A18) : 3050


In [21]:
# US-06 - Étape 6 : Souscripteurs sans téléphone (A03)

sans_tel = df.loc[df["type_personne"] == "souscripteur", "telephone"].fillna("").str.strip() == ""
print("Souscripteurs sans téléphone (A03) :", sans_tel.sum())

Souscripteurs sans téléphone (A03) : 5989


In [22]:
# US-06 - Étape 7 : E-mails déjà vus sur un autre client (A05)

print("E-mails déjà vus sur un autre client (A05) :", df["email"].dropna().duplicated().sum())

E-mails déjà vus sur un autre client (A05) : 2840


In [23]:
# US-06 - Étape 8 : Doublons stricts nom + prénom + naissance (A15)

print("Doublons stricts nom + prénom + naissance (A15) :", df.dropna(subset=["nom", "prenom", "date_naissance"]).duplicated(subset=["nom", "prenom", "date_naissance"]).sum())

Doublons stricts nom + prénom + naissance (A15) : 738


In [24]:
# US-06 - Étape 9 : Retrouver le cas Meunier

meunier = df[df["client_id"].isin(["A-4471", "H-9012", "S-2208"])]
meunier[["client_id", "raison_sociale", "adresse", "cp", "ville", "telephone"]]

,client_id,raison_sociale,adresse,cp,ville,telephone
34449,A-4471,Société Meunier SARL,27 rue Pasteur,69003,Lyon,0478473441
73792,H-9012,MEUNIER SARL,27 rue Pasteur,69003,Lyon,0409980316
88068,S-2208,Sté Meunier S.A.R.L.,27 rue Pasteur,69003,Lyon,0417379714


## US-07

In [25]:
# US-07 - Étape 1 : Motifs de code postal, téléphone et pays

def motif(s):
    """9 remplace un chiffre, A une lettre : on voit la forme d'une valeur, pas la valeur."""
    return s.fillna("<NULL>").str.replace(r"[0-9]", "9", regex=True).str.replace(r"[A-Za-z]", "A", regex=True)

for colonne in ["cp", "telephone", "pays"]:
    print(f"\nMotifs de {colonne} :", motif(df[colonne]).value_counts().head(12))


Motifs de cp : cp
99999       97683
9999         2068
AA9A 9AA      125
9999-999      124
Name: count, dtype: int64

Motifs de telephone : telephone
9999999999           66638
<AAAA>                5999
+99999999999          5301
99-99-99-99-99        4267
99.99.99.99.99        4261
99 99 99 99 99        4243
+99 9 99 99 99 99     4235
                      2000
                      2000
+999999999999          338
999999999              273
9999A99999             250
Name: count, dtype: int64

Motifs de pays : pays
AA        97400
AAAAAA     1600
A           600
            400
Name: count, dtype: int64


In [29]:
# US-07 - Étape 2 : Conformité des codes postaux français (RQ-005)

def conformite(serie, regex):
    """(testées, en échec) pour une expression régulière complète ; une valeur nulle est en échec."""
    ok = serie.fillna("").str.fullmatch(regex)
    return len(serie), int((~ok).sum())

fr = df[df.pays == "FR"]
n, ko = conformite(fr.cp, r"[0-9]{5}")
print(f"\nRQ-005 : code postal à 5 chiffres, adresses FR : {ko} écarts sur {n} ({100*ko/n:.3f} %)")

print("A01 (zéro initial perdu) :\n", fr[fr.cp.fillna("").str.fullmatch(r"[0-9]{4}")].groupby(["cp", "ville"]).size().sort_values(ascending=False).head(10))


RQ-005 : code postal à 5 chiffres, adresses FR : 1456 écarts sur 95437 (1.526 %)
A01 (zéro initial perdu) :
 cp    ville            
1000  Bourg-en-Bresse      92
1100  Oyonnax              77
7100  Annonay              71
1700  Miribel              69
7200  Aubenas              67
1500  Ambérieu-en-Bugey    63
1120  Montluel             53
3200  Vichy                52
3000  Moulins              50
1360  Loyettes             44
dtype: int64


In [31]:
# US-07 - Étape 3 : Exemples de formats de téléphone

for m in motif(df["telephone"]).value_counts().head(12).index:
    exemples = df.loc[motif(df["telephone"]) == m, "telephone"].dropna().head(3).tolist()
    print(f"{m} : {exemples}")

9999999999 : ['0346234195', '0212942708', '0129310237']
<AAAA> : []
+99999999999 : ['+33582679882', '+32298071353', '+33637433839']
99-99-99-99-99 : ['06-16-54-95-35', '07-40-61-91-78', '06-21-17-38-59']
99.99.99.99.99 : ['06.72.90.03.14', '09.41.88.86.22', '04.41.93.06.69']
99 99 99 99 99 : ['01 65 06 46 10', '07 18 72 38 48', '06 14 73 08 41']
+99 9 99 99 99 99 : ['+33 6 02 05 35 92', '+33 6 32 75 53 35', '+33 3 27 74 39 34']
 : ['', '', '']
    : ['   ', '   ', '   ']
+999999999999 : ['+351239622910', '+352040710040', '+352805567541']
999999999 : ['044915574', '061411425', '042919809']
9999A99999 : ['0618O36359', '0297O94510', '0790O34277']


In [33]:
# US-07 - Étape 4 : Conformité des téléphones
# Regex proposée : formats français et internationaux

regex_telephone = r"(?:0\d(?:[ .-]?\d{2}){4}|\+\d{1,3}[ .-]?(?:\d[ .-]?){6,12}\d)"

renseignes = df.telephone.notna() & (df.telephone.str.strip() != "")

if regex_telephone:
    n, ko = conformite(df.loc[renseignes, "telephone"], regex_telephone)
    print(f"Téléphone : {ko} écarts sur {n} numéros renseignés ({100*ko/n:.3f} %)")

Téléphone : 886 écarts sur 90001 numéros renseignés (0.984 %)


In [34]:
# US-07 - Étape 5 : Pays présents dans le référentiel (RQ-007)

ref_pays = set(lire("SELECT code_pays FROM silver.ref_pays").code_pays)
hors = df[~df.pays.isin(ref_pays)]

print(f"Pays hors référentiel : {len(hors)} sur {len(df)}")
print(hors.pays.value_counts(dropna=False).head(15))

Pays hors référentiel : 3050 sur 100000
pays
France    1200
F          600
           400
fr         400
Frnace     400
UK          20
ZZ          10
EN          10
XX          10
Name: count, dtype: int64


## US-04

In [38]:
# US-04 - Préparation : rendre disponible la fonction motifs du professeur
motifs = motif

In [39]:
# US-04 - Étape 1 : Profilage des colonnes anonymes du CIF

cif = lire("SELECT * FROM bronze.cif_clients WHERE _batch_id = :b", b=BATCH)
cif = cif[[c for c in cif.columns if c.startswith("column")]]

dico = pd.DataFrame({
    "nuls": cif.isna().sum(),
    "distincts": cif.nunique(),
    "long_min": cif.apply(lambda s: s.str.len().min()).astype("Int64"),
    "long_max": cif.apply(lambda s: s.str.len().max()).astype("Int64"),
    "motif_dominant": cif.apply(lambda s: motifs(s).value_counts().index[0]),
    "exemples": cif.apply(lambda s: " | ".join(s.dropna().sample(3, random_state=1).tolist()))
})

print(dico.to_string())

          nuls  distincts  long_min  long_max      motif_dominant                                                                          exemples
column0      0      40000         7         7             A999999                                                       K186124 | K388169 | K436056
column1      0        731         3        31              AAAAAA                                                      GAILLARD | RIVIERE | LE GALL
column2    645        180         3        10              AAAAAA                                                     Nathalie | Samira | Charlotte
column3    645      21081        10        10          99/99/9999                                              15/02/1969 | 26/04/2007 | 04/12/1959
column4      0       9811        11        31  99 AAA AA AAAAAAAA               49 AVENUE DE LA LIBÉRATION | 95 rue des Écoles | 20 rue de Bretagne
column5      0        188         4         8               99999                                               

In [40]:
# US-04 - Étape 2 : Recherche de la clé entre CIF clients et comptes

compte = lire("SELECT * FROM bronze.cif_compte WHERE _batch_id = :b", b=BATCH)
compte = compte[[c for c in compte.columns if c.startswith("column")]]

liens = [(a, k, round(100 * compte[a].isin(set(cif[k].dropna())).mean(), 1)) for a in compte.columns for k in cif.columns]

print(pd.DataFrame(liens, columns=["colonne_compte", "colonne_clients", "pct_retrouves"]).sort_values("pct_retrouves", ascending=False).head(5))

   colonne_compte colonne_clients  pct_retrouves
10        column1         column0          100.0
21        column2         column1            0.0
23        column2         column3            0.0
24        column2         column4            0.0
25        column2         column5            0.0


## US-09 : Trois définitions

In [43]:
# US-09 - Étape 1 : Chargement et préparation des contrats

contrat = lire("SELECT souscripteur_id, date_debut, date_fin, date_derniere_cotisation FROM bronze.crm_contrat WHERE _batch_id = :b", b=BATCH)

for c in ["date_debut", "date_fin", "date_derniere_cotisation"]:
    contrat[c] = pd.to_datetime(contrat[c], errors="coerce")

ref, cotisation = pd.Timestamp(DATE_REF), pd.Timestamp(COTISATION_MIN)

print("date de référence :", ref.date(), "| cotisation depuis :", cotisation.date())


date de référence : 2026-10-01 | cotisation depuis : 2026-07-01


In [44]:
# US-09 - Étape 2 : Calcul des clients actifs selon les trois directions

# Contrat en cours à la date de référence
en_cours = (contrat.date_debut <= ref) & (contrat.date_fin >= ref)

# Direction commerciale : au moins un contrat
n_commerciale = contrat.souscripteur_id.nunique()

# Direction des sinistres : au moins un contrat en cours
n_sinistres = contrat.loc[en_cours, "souscripteur_id"].nunique()

# Direction des risques : contrat en cours et cotisation à jour
n_risques = contrat.loc[
    en_cours & (contrat.date_derniere_cotisation >= cotisation),
    "souscripteur_id"
].nunique()

print("Commerciale :", n_commerciale)
print("Sinistres :", n_sinistres)
print("Risques (CD-001) :", n_risques)


Commerciale : 50508
Sinistres : 44116
Risques (CD-001) : 38049
